# EuroMetrics – European Economic Analysis

## Research questions

1. Has the economic gap between Eastern and Western Europe narrowed since 2000?
2. Which EU countries have lost the most relative economic position since 2000, and how has Sweden performed?

## Data collection from MariaDB

In [1]:
import sys
import pandas as pd

sys.path.append("../src")

from database import connect_database
from queries import GET_PPS_DATA

db = connect_database()
cursor = db.cursor()

cursor.execute(GET_PPS_DATA)

rows = cursor.fetchall()

columns = [column[0] for column in cursor.description]

df = pd.DataFrame(rows, columns=columns)

cursor.close()
db.close()

df.head()

,country_name,region,year,pps_index,quality_flag
0,Austria,West,2000,132.000000,NaN
1,Austria,West,2001,127.000000,NaN
2,Austria,West,2002,127.000000,NaN
3,Austria,West,2003,129.000000,NaN
4,Austria,West,2004,129.000000,NaN


## Exploratory Data Analysis

In [2]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())

(702, 5)
country_name       str
region             str
year             int64
pps_index       object
quality_flag       str
dtype: object
country_name      0
region            0
year              0
pps_index         0
quality_flag    683
dtype: int64


In [3]:
# Convert PPS index to a numeric data type
df["pps_index"] = df["pps_index"].astype(float)

print(df.dtypes)

country_name        str
region              str
year              int64
pps_index       float64
quality_flag        str
dtype: object


## Question 1 – Has the economic gap between Eastern and Western Europe narrowed since 2000?

### Mean and median

In [4]:
regional_stats = df.groupby(["region", "year"], as_index=False).agg(
    mean_pps=("pps_index", "mean"),
    median_pps=("pps_index", "median")
)

regional_stats

,region,year,mean_pps,median_pps
0,East,2000,47.818182,48.0
1,East,2001,49.636364,48.0
2,East,2002,51.545455,49.0
3,East,2003,54.181818,53.0
4,East,2004,56.090909,56.0
...,...,...,...,...
99,West,2021,154.142857,122.0
100,West,2022,154.142857,123.0
101,West,2023,150.428571,121.0
102,West,2024,149.857143,119.0


### Economic gap between East and West

In [5]:
regional_gap = regional_stats.pivot(
    index="year",
    columns="region",
    values="mean_pps"
)

regional_gap["gap"] = regional_gap["West"] - regional_gap["East"]

regional_gap.round(2)

region,East,North,South,West,gap
year,,,,,
2000,47.82,128.00,94.67,146.86,99.04
2001,49.64,125.00,94.67,145.29,95.65
2002,51.55,123.67,95.00,145.71,94.17
2003,54.18,123.00,96.50,145.14,90.96
2004,56.09,125.67,96.00,146.14,90.05
2005,58.27,124.33,96.00,147.29,89.01
2006,59.73,124.00,96.17,148.29,88.56
2007,62.64,125.67,95.33,147.86,85.22
2008,64.91,125.67,95.17,146.29,81.38


In [6]:
gap_2000 = regional_gap.loc[2000, "gap"]
gap_2025 = regional_gap.loc[2025, "gap"]

gap_reduction = gap_2000 - gap_2025
percentage_reduction = gap_reduction / gap_2000 * 100

print(f"Economic gap in 2000: {gap_2000:.2f}")
print(f"Economic gap in 2025: {gap_2025:.2f}")
print(f"Reduction: {gap_reduction:.2f} index points")
print(f"Percentage reduction: {percentage_reduction:.2f}%")

Economic gap in 2000: 99.04
Economic gap in 2025: 71.17
Reduction: 27.87 index points
Percentage reduction: 28.14%


### Median gap analysis

In [7]:
median_gap = regional_stats.pivot(
    index="year",
    columns="region",
    values="median_pps"
)

median_gap["gap"] = median_gap["West"] - median_gap["East"]

median_gap.loc[[2000, 2025]].round(2)

region,East,North,South,West,gap
year,,,,,
2000,48.0,129.0,90.5,132.0,84.0
2025,78.0,111.0,94.0,118.0,40.0


## Question 2 – Which EU countries have lost the most relative economic position since 2000, and how has Sweden performed?

### Change in PPS index between 2000 and 2025

In [8]:
country_change = df.pivot(
    index="country_name",
    columns="year",
    values="pps_index"
)

country_change = country_change[[2000, 2025]].copy()

country_change["change"] = country_change[2025] - country_change[2000]

country_change = country_change.sort_values("change")

country_change

year,2000,2025,change
country_name,,,
Italy,122.0,96.0,-26.0
Sweden,134.0,111.0,-23.0
France,118.0,98.0,-20.0
Finland,121.0,101.0,-20.0
Greece,85.0,68.0,-17.0
Austria,132.0,118.0,-14.0
Netherlands,144.0,133.0,-11.0
Belgium,125.0,114.0,-11.0
Germany,125.0,115.0,-10.0


### Sweden's economic development

In [9]:
sweden = df[df["country_name"] == "Sweden"]

sweden = sweden.sort_values("year")

sweden[["year", "pps_index"]]

,year,pps_index
624,2000,134.0
625,2001,129.0
626,2002,127.0
627,2003,128.0
628,2004,130.0
629,2005,127.0
630,2006,128.0
631,2007,131.0
632,2008,128.0
633,2009,125.0
